# Kata 12 — Prompt Chaining Multi-Pass

> Spec: [`specs/012-prompt-chaining/spec.md`](../../specs/012-prompt-chaining/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=15)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Para tareas grandes (auditar 10 archivos, resumir 50 páginas), descompongo en **pases secuenciales**: pase local por unidad con salida tipada, luego pase de integración que sólo ve los outputs locales — nunca los crudos.

## 2. Por qué importa

Concatenar todo en un sólo prompt satura atención: respuestas superficiales, alucinaciones cruzadas, costo máximo, calidad mínima.

## 3. Ejemplo correcto — fan-out + integración

In [ ]:
import json

DOCS = {
    "ticket-A": "Cliente VIP-001 reporta latencia >5s en checkout. Pasó tres veces esta semana.",
    "ticket-B": "Pago rechazado en Atlas tier 2. Mensaje de error: 'AUTH-503'.",
    "ticket-C": "Solicita feature: integración con Slack para alertas. Tier 3.",
    "ticket-D": "Reporta bug en facturación: monto duplicado en factura mensual.",
}

PER_DOC_TOOL = {
    "name": "submit_ticket_summary",
    "description": "Resumen tipado del ticket.",
    "input_schema": {
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string"},
            "category": {"type": "string", "enum": ["bug", "perf", "billing", "feature", "auth", "other"]},
            "severity": {"type": "string", "enum": ["low", "medium", "high"]},
            "one_liner": {"type": "string"},
        },
        "required": ["ticket_id", "category", "severity", "one_liner"],
    },
}

def pass1_per_doc(ticket_id, text):
    resp = client.messages.create(
        system="Resume el ticket en formato tipado.",
        messages=[{"role":"user","content": f"ID: {ticket_id}\n{text}"}],
        tools=[PER_DOC_TOOL],
        tool_choice={"type":"tool","name":"submit_ticket_summary"},
    )
    return next(b for b in resp.content if b.type == "tool_use").input

local = [pass1_per_doc(k, v) for k, v in DOCS.items()]
for r in local: print(r)

In [ ]:
def pass2_integrate(local_summaries):
    payload = json.dumps(local_summaries, ensure_ascii=False, indent=2)
    resp = client.messages.create(
        system="Recibirás resúmenes tipados de tickets. Identifica la categoría con mayor severidad acumulada y recomienda 1 acción.",
        messages=[{"role":"user","content": payload}],
    )
    return next((b.text for b in resp.content if b.type == "text"), "")

print(pass2_integrate(local))

## 4. Anti-patrón — single-pass con todo concatenado

In [ ]:
big = "\n\n".join(f"=== {k} ===\n{v}" for k, v in DOCS.items())
resp_bad = client.messages.create(
    system="Eres un analista. Identifica patrones, severidad y recomienda acciones.",
    messages=[{"role":"user","content": big}],
)
print(next((b.text for b in resp_bad.content if b.type == "text"), "")[:600])
print("\ninput_tokens:", resp_bad.usage.input_tokens)

En 4 docs el monolítico todavía funciona. En 50 colapsa: omite docs, mezcla detalles, no produce salida tipada agregable.

## 5. Argumento de certificación

- **Pase 1 con salida tipada por schema** (Kata 5).
- **Pase 2 ve sólo lo tipado**, jamás los docs crudos.
- **Paralelizable**: el pase 1 corre con los subagentes del Kata 4.
- **Atención focalizada por unidad** = menos alucinación cruzada.
- **Conexión con otros katas**: encaja con Kata 4 (subagentes) y Kata 11 (cada pase respeta el límite de contexto).

## 6. Auto-evaluación

**1. ¿Cuándo NO conviene chaining (overhead > beneficio)?**

Cuando los docs son muy pocos y muy cortos (<2-3 docs cabiendo holgadamente). El overhead de N llamadas no se justifica.

**2. ¿Qué pasa si un pase 1 falla en una unidad? ¿se aborta, se omite, se reintenta?**

Reintenta una vez (transitorio); si vuelve a fallar, lo registra con `error: true` en el agregado y deja que el pase 2 lo refleje en el reporte. Nunca se omite silenciosamente.

**3. ¿Cómo evito que el pase 2 "rellene" lo que el pase 1 dejó vacío?**

El system prompt del pase 2 es explícito: "no inventes información que no aparezca en los resúmenes". Y al ser tipados, el pase 2 puede ver `null` o `unclear` y respetarlos.